# TCN denoising autoencoder for synthetic SCG

Generate paired **corrupted input → its own clean target** using the five corruption families in
`noise reduction hands-on.ipynb`. Randomize signal morphology, noise combinations, and severity.
The noncausal TCN encoder–decoder uses past and future context for offline reconstruction.

Run from the SensorAI checkout. Install the repository `requirements.txt` and a server-compatible
PyTorch build in the notebook kernel. No training runs unless `RUN_TRAINING = True` below.
Missing timestamps must be known: missing samples are interpolated and accompanied by a validity
channel. Unknown timestamp loss/resampling from the reference notebook is excluded because it
changes temporal alignment. Reconstruction inside long gaps is a model estimate, not observed data.

In [ ]:
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
import copy
import io
import json
import random

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, savgol_filter
import sys

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "lib" / "dsp.py").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from lib.dsp import scg_simulate

CFG = {
    "seed": 42, "fs": 100, "duration": 10,
    "train_clean": 4096, "val_clean": 512, "test_clean": 512,
    "train_variants": 4, "eval_variants": 2,
    "batch_size": 32, "epochs": 100, "lr": 1e-3,
    "weight_decay": 1e-4, "patience": 15, "grad_clip": 1.0,
    "slope_weight": 0.1, "clean_probability": 0.05,
    "channels": 32, "latent_channels": 16, "dropout": 0.1,
    "dilations": [1, 2, 4, 8, 16, 32],
}
# Noise amplitudes are relative to clean signal standard deviation unless noted.
NOISE = {
    "band_low_hz": (25.0, 33.0), "band_high_hz": (38.0, 47.0),
    "band_amplitude": (0.1, 2.0),
    "drift_amplitude": (0.2, 2.0),
    "drift_slow_hz": (0.05, 0.2), "drift_fast_hz": (0.2, 0.6),
    "spike_count": (1, 8), "spike_peak_multiple": (2.0, 50.0),
    "gap_count": (1, 3), "gap_seconds": (0.01, 0.7),
    "motion_count": (1, 3), "motion_seconds": (0.3, 3.0),
    "motion_slow_hz": (0.2, 1.0), "motion_fast_hz": (25.0, 40.0),
    "motion_amplitude": (0.5, 6.0),
}
NOISE_NAMES = ("baseline_drift", "band_noise", "spikes", "signal_loss", "motion")
N = int(CFG["fs"] * CFG["duration"])
OUTPUT_DIR = REPO_ROOT / "outputs" / "tcn_denoising_autoencoder"
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_PREVIEW = False
assert N >= 32 and CFG["duration"] >= 3
assert 0 < NOISE["band_low_hz"][0] < NOISE["band_low_hz"][1] < NOISE["band_high_hz"][0]
assert max(NOISE["band_high_hz"][1], NOISE["motion_fast_hz"][1]) < CFG["fs"] / 2

## 1. Independent clean signal banks

Reuse `scg_simulate` with noise disabled, varying heart rate, blood pressure and wavelet family.
Center and standardize synthetic targets into arbitrary amplitude units, then randomize gain and
phase. Train/validation/test use separate seed namespaces; variants of a base waveform stay in
one split. This is a synthetic benchmark, not a patient-level real-data validation.

In [ ]:
def make_clean_signal(seed, cfg):
    rng = np.random.default_rng(seed)
    fs, duration = cfg["fs"], cfg["duration"]
    n = int(fs * duration)
    hr, rr = int(rng.integers(50, 111)), int(rng.integers(10, 25))
    sbp, dbp = int(rng.integers(95, 151)), int(rng.integers(60, 96))
    pulse = str(rng.choice(["db", "sym", "coif"]))
    # lib.dsp uses both global RNGs and resets NumPy's seed internally.
    # Restore both states so generation cannot perturb training randomness.
    python_state, numpy_state = random.getstate(), np.random.get_state()
    try:
        random.seed(seed)
        np.random.seed(seed)
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            raw = scg_simulate(
                num_rows=1, duration=duration + 2, sampling_rate=fs,
                heart_rate=(hr, hr), respiratory_rate=(rr, rr),
                systolic=(sbp, sbp), diastolic=(dbp, dbp), pulse_type=pulse,
                add_respiratory=False, noise_amplitude=0,
                random_state=seed, silent=True,
            )[:int((duration + 2) * fs)]  # exclude the six metadata values
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)
    start = int(rng.integers(0, len(raw) - n + 1))
    clean = np.asarray(raw[start:start + n], dtype=np.float64)
    clean -= clean.mean()
    if not np.isfinite(clean).all() or clean.std() <= 1e-15:
        raise ValueError("Simulator returned a degenerate signal")
    gain = float(rng.uniform(0.5, 2.0))
    clean = clean / clean.std() * gain
    return clean.astype(np.float32)


def make_clean_bank(split, cfg):
    split_id = {"train": 1, "val": 2, "test": 3}[split]
    count = cfg[f"{split}_clean"]
    # Disjoint scalar seeds, including across splits.
    if count >= 1_000_000:
        raise ValueError("Use fewer than 1,000,000 clean records per split")
    seeds = cfg["seed"] + split_id * 1_000_000 + np.arange(count)
    return np.stack([make_clean_signal(int(seed), cfg) for seed in seeds])

## 2. Mixed corruptions and input-only normalization

Sample uniformly from all 31 nonempty subsets of the five noise families (plus 5% clean examples).
Apply additive corruptions first, then replace spike samples, then apply signal loss. Each selected
family draws fresh parameters. Validation/test seeds remain fixed; training noise changes by epoch.
Normalization uses only observed noisy samples (median and IQR), and applies the same transform
to the target. No clean statistics are required at inference. Spike amplitudes reach the reference
notebook's 50× peak example; band noise follows its Butterworth-filtered white-noise construction.

In [ ]:
def corrupt_signal(clean, rng, cfg, noise_cfg, forced_combo=None):
    clean = np.asarray(clean, dtype=np.float64)
    n, fs = len(clean), cfg["fs"]
    t = np.arange(n) / fs
    sd = max(float(clean.std()), 1e-8)
    noisy, valid = clean.copy(), np.ones(n, dtype=bool)
    u = lambda name: float(rng.uniform(*noise_cfg[name]))
    count = lambda name: int(rng.integers(noise_cfg[name][0], noise_cfg[name][1] + 1))
    if forced_combo is None:
        combo = 0 if rng.random() < cfg["clean_probability"] else int(rng.integers(1, 32))
    else:
        combo = int(forced_combo)
        if not 0 <= combo <= 31:
            raise ValueError("forced_combo must be in [0, 31]")
    details = {"combo": combo}

    if combo & 1:
        f1, f2, amplitude = u("drift_slow_hz"), u("drift_fast_hz"), u("drift_amplitude")
        walk = np.cumsum(rng.standard_normal(n))
        window = min(int(3 * fs) | 1, n if n % 2 else n - 1)
        walk = savgol_filter(walk, window_length=window, polyorder=2)
        walk = (walk - walk.mean()) / (walk.std() + 1e-8)
        trend = (t / max(t[-1], 1 / fs) - 0.5) ** 2
        drift = (0.85 * np.sin(2 * np.pi * f1 * t + rng.uniform(0, 2 * np.pi))
                 + 0.55 * np.sin(2 * np.pi * f2 * t + rng.uniform(0, 2 * np.pi))
                 + 0.6 * (trend - trend.mean()) + 0.3 * walk)
        noisy += sd * amplitude * drift
        details["drift"] = [f1, f2, amplitude]
    if combo & 2:
        low, high, amplitude = u("band_low_hz"), u("band_high_hz"), u("band_amplitude")
        sos = butter(3, [low, high], btype="bandpass", fs=fs, output="sos")
        noisy += sosfilt(sos, rng.normal(0, sd * amplitude, n))
        details["band"] = [low, high, amplitude]
    if combo & 16:
        details["motion"] = []
        for _ in range(count("motion_count")):
            length = min(n - 1, max(3, int(u("motion_seconds") * fs)))
            start = int(rng.integers(0, n - length + 1))
            time = np.arange(length) / fs
            slow, fast, amplitude = u("motion_slow_hz"), u("motion_fast_hz"), u("motion_amplitude")
            artifact = np.hanning(length) * (
                np.sin(2 * np.pi * slow * time + rng.uniform(0, 2 * np.pi))
                + 0.4 * np.sin(2 * np.pi * fast * time + rng.uniform(0, 2 * np.pi))
                + 0.04 * rng.standard_normal(length))
            noisy[start:start + length] += sd * amplitude * artifact
            details["motion"].append([start, length, slow, fast, amplitude])
    if combo & 4:
        positions = rng.choice(n, size=min(n, count("spike_count")), replace=False)
        multiples = rng.uniform(*noise_cfg["spike_peak_multiple"], size=len(positions))
        noisy[positions] = rng.choice([-1, 1], size=len(positions)) * multiples * np.max(np.abs(clean))
        details["spikes"] = positions.tolist()
    if combo & 8:
        # Keep boundaries observed to ensure interpolation is defined.
        details["gaps"] = []
        for _ in range(count("gap_count")):
            length = min(n // 4, max(1, int(u("gap_seconds") * fs)))
            start = int(rng.integers(1, n - length))
            valid[start:start + length] = False
            details["gaps"].append([start, length])
        noisy[~valid] = np.nan
    return noisy, valid, details


def prepare_input(noisy, valid=None):
    noisy = np.asarray(noisy, dtype=np.float64)
    if noisy.ndim != 1 or len(noisy) < 32:
        raise ValueError("Expected a 1-D signal with at least 32 samples")
    if valid is not None and np.shape(valid) != noisy.shape:
        raise ValueError("valid must have the same shape as noisy")
    valid = np.isfinite(noisy) if valid is None else np.asarray(valid, bool) & np.isfinite(noisy)
    if valid.sum() < 2:
        raise ValueError("At least two observed samples are required")
    observed = noisy[valid]
    offset = float(np.median(observed))
    q25, q75 = np.percentile(observed, [25, 75])
    scale = max(float((q75 - q25) / 1.349), 1e-6)
    filled = np.interp(np.arange(len(noisy)), np.flatnonzero(valid), observed)
    x = np.stack([(filled - offset) / scale, valid.astype(float)]).astype(np.float32)
    return x, offset, scale


if RUN_PREVIEW:
    preview_clean = make_clean_signal(CFG["seed"], CFG)
    preview_noisy, preview_valid, preview_details = corrupt_signal(
        preview_clean, np.random.default_rng(123), CFG, NOISE, forced_combo=31)
    plt.figure(figsize=(12, 3))
    plt.plot(np.arange(N) / CFG["fs"], preview_clean, label="Clean target")
    plt.plot(np.arange(N) / CFG["fs"], preview_noisy, alpha=0.6, label="All five corruptions")
    plt.xlabel("Time (s)"); plt.legend(); plt.tight_layout(); plt.show()
    print(preview_details)

## 3. Reproducible online dataset

Only clean banks are stored in memory. Corruptions are generated on demand from split, epoch,
and sample index, independent of DataLoader order. `num_workers=0` keeps notebook execution
portable. A clean record has multiple noise variants per epoch.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class MixedNoiseDataset(Dataset):
    def __init__(self, bank, split, cfg, noise_cfg):
        self.bank, self.split = bank, split
        self.cfg, self.noise_cfg = copy.deepcopy(cfg), copy.deepcopy(noise_cfg)
        self.epoch = 0
        self.variants = cfg["train_variants"] if split == "train" else cfg["eval_variants"]
        self.split_id = {"train": 1, "val": 2, "test": 3}[split]

    def __len__(self):
        return len(self.bank) * self.variants

    def set_epoch(self, epoch):
        self.epoch = epoch if self.split == "train" else 0

    def __getitem__(self, index):
        rng = np.random.default_rng(np.random.SeedSequence(
            [self.cfg["seed"], self.split_id, self.epoch, int(index), 101]))
        clean = self.bank[index // self.variants]
        noisy, valid, details = corrupt_signal(clean, rng, self.cfg, self.noise_cfg)
        x, offset, scale = prepare_input(noisy, valid)
        target = ((clean - offset) / scale).astype(np.float32)[None, :]
        return {
            "x": torch.from_numpy(x), "y": torch.from_numpy(target),
            "offset": np.float32(offset), "scale": np.float32(scale),
            "combo": details["combo"],
        }


def make_loader(dataset, cfg, shuffle=False):
    generator = torch.Generator().manual_seed(cfg["seed"])
    return DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=shuffle,
                      num_workers=0, pin_memory=DEVICE.type == "cuda", generator=generator)

## 4. TCN encoder–decoder

Two strided convolutions compress time by 4×, followed by a dilated residual TCN and a channel
bottleneck. Interpolation and convolution decode to the original length. There is no direct
input-to-output skip, so the model must reconstruct through the latent representation.
Symmetric padding makes this an offline, noncausal model. The output activation is linear.

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.GELU(), nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class TCNDenoisingAutoencoder(nn.Module):
    def __init__(self, channels=32, latent_channels=16, dilations=(1, 2, 4, 8, 16, 32), dropout=0.1):
        super().__init__()
        self.stem = nn.Conv1d(2, channels, 7, padding=3)
        self.down1 = nn.Conv1d(channels, channels, 5, stride=2, padding=2)
        self.down2 = nn.Conv1d(channels, channels * 2, 5, stride=2, padding=2)
        self.tcn = nn.Sequential(*[TemporalBlock(channels * 2, d, dropout) for d in dilations])
        self.to_latent = nn.Conv1d(channels * 2, latent_channels, 1)
        self.from_latent = nn.Conv1d(latent_channels, channels * 2, 1)
        self.up1 = nn.Conv1d(channels * 2, channels, 5, padding=2)
        self.refine = TemporalBlock(channels, 1, dropout)
        self.output = nn.Conv1d(channels, 1, 7, padding=3)

    def forward(self, x):
        length = x.shape[-1]
        h = F.gelu(self.stem(x))
        h = F.gelu(self.down1(h))
        half_length = h.shape[-1]
        h = F.gelu(self.down2(h))
        z = self.to_latent(self.tcn(h))
        h = F.gelu(self.from_latent(z))
        h = F.interpolate(h, size=half_length, mode="linear", align_corners=False)
        h = F.gelu(self.up1(h))
        h = F.interpolate(h, size=length, mode="linear", align_corners=False)
        return self.output(self.refine(h))


def model_kwargs(cfg):
    return {key: cfg[key] for key in ("channels", "latent_channels", "dilations", "dropout")}


def reconstruction_loss(prediction, target, slope_weight):
    mse = F.mse_loss(prediction, target)
    slope = F.mse_loss(prediction.diff(dim=-1), target.diff(dim=-1))
    return mse + slope_weight * slope

## 5. Training, validation, and checkpoints (disabled by default)

Loss = sample MSE + 0.1 × first-difference MSE. Validate on a fixed corruption set, reduce the
learning rate on a plateau, clip gradients, and early-stop. Save best and last checkpoints,
configuration, clean banks, and epoch history. Test data is not used for model selection.
Set `RUN_TRAINING = True` in the configuration cell on the server, then run all cells.
Use a new `OUTPUT_DIR` for a new experiment; existing checkpoints are protected from overwrite.

In [ ]:
def run_epoch(model, loader, cfg, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_count = 0.0, 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            x, y = batch["x"].to(DEVICE), batch["y"].to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            prediction = model(x)
            loss = reconstruction_loss(prediction, y, cfg["slope_weight"])
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss; inspect input scaling and noise ranges")
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                optimizer.step()
            total_loss += float(loss.detach()) * len(x)
            total_count += len(x)
    return total_loss / total_count


def train_model(cfg, noise_cfg, output_dir):
    output_dir = Path(output_dir)
    if any((output_dir / name).exists() for name in ("best.pt", "last.pt")):
        raise FileExistsError("Choose a new OUTPUT_DIR to preserve this experiment")
    seed_everything(cfg["seed"])
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "config.json").write_text(json.dumps({"cfg": cfg, "noise": noise_cfg}, indent=2))
    print("Generating independent clean banks...")
    banks = {split: make_clean_bank(split, cfg) for split in ("train", "val", "test")}
    np.savez_compressed(output_dir / "clean_banks.npz", **banks)
    train_set = MixedNoiseDataset(banks["train"], "train", cfg, noise_cfg)
    val_set = MixedNoiseDataset(banks["val"], "val", cfg, noise_cfg)
    train_loader, val_loader = make_loader(train_set, cfg, True), make_loader(val_set, cfg)
    model = TCNDenoisingAutoencoder(**model_kwargs(cfg)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5)
    history, best_val, stale = [], float("inf"), 0
    print(f"Device: {DEVICE}; parameters: {sum(p.numel() for p in model.parameters()):,}")
    for epoch in range(cfg["epochs"]):
        train_set.set_epoch(epoch)
        train_loss = run_epoch(model, train_loader, cfg, optimizer)
        val_loss = run_epoch(model, val_loader, cfg)
        scheduler.step(val_loss)
        improved = val_loss < best_val
        if improved:
            best_val, stale = val_loss, 0
        else:
            stale += 1
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
                        "lr": optimizer.param_groups[0]["lr"]})
        checkpoint = {
            "epoch": epoch + 1, "model": model.state_dict(),
            "model_kwargs": model_kwargs(cfg), "cfg": cfg, "noise_cfg": noise_cfg,
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "best_val": best_val, "history": history,
        }
        torch.save(checkpoint, output_dir / "last.pt")
        if improved:
            torch.save(checkpoint, output_dir / "best.pt")
        (output_dir / "history.json").write_text(json.dumps(history, indent=2))
        print(f"Epoch {epoch + 1:03d}: train={train_loss:.6f}, val={val_loss:.6f}, best={best_val:.6f}")
        if stale >= cfg["patience"]:
            print("Early stopping")
            break
    return history


if RUN_TRAINING:
    history = train_model(CFG, NOISE, OUTPUT_DIR)
else:
    print("Training disabled. Set RUN_TRAINING=True on the server when ready.")

## 6. Held-out recovery, metrics and inference

Reload the best checkpoint and its configuration. Report per-record RMSE, SNR and correlation
before/after recovery, grouped by corruption combination; also report RMSE inside known gaps.
SNR uses the clean target energy and is capped numerically at negligible errors. The input
baseline is the interpolated noisy signal supplied to the model. Evaluate once after choosing
the model using validation only. `recover_signal` accepts one sampled window at the training
sampling rate; values are returned in the input's amplitude units.

In [ ]:
def load_trained_model(checkpoint_path, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = TCNDenoisingAutoencoder(**checkpoint["model_kwargs"]).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model, checkpoint


@torch.no_grad()
def recover_signal(model, noisy, valid=None):
    x, offset, scale = prepare_input(noisy, valid)
    device = next(model.parameters()).device
    model.eval()
    prediction = model(torch.from_numpy(x[None]).to(device))[0, 0].cpu().numpy()
    return prediction * scale + offset


def signal_metrics(target, prediction):
    error = np.mean((prediction - target) ** 2, axis=1)
    power = np.mean(target ** 2, axis=1)
    a, b = target - target.mean(axis=1, keepdims=True), prediction - prediction.mean(axis=1, keepdims=True)
    correlation = np.sum(a * b, axis=1) / np.maximum(
        np.sqrt(np.sum(a * a, axis=1) * np.sum(b * b, axis=1)), 1e-12)
    return {"rmse": np.sqrt(error),
            "snr_db": 10 * np.log10(np.maximum(power, 1e-12) / np.maximum(error, 1e-12)),
            "correlation": correlation}


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    device = next(model.parameters()).device
    rows = []
    for batch in loader:
        prediction = model(batch["x"].to(device)).cpu().numpy()[:, 0]
        scale, offset = batch["scale"].numpy()[:, None], batch["offset"].numpy()[:, None]
        target = batch["y"].numpy()[:, 0] * scale + offset
        baseline = batch["x"].numpy()[:, 0] * scale + offset
        prediction = prediction * scale + offset
        before, after = signal_metrics(target, baseline), signal_metrics(target, prediction)
        for i, combo in enumerate(batch["combo"].tolist()):
            row = {"combo": combo}
            row.update({f"input_{k}": float(v[i]) for k, v in before.items()})
            row.update({f"output_{k}": float(v[i]) for k, v in after.items()})
            row["snr_gain_db"] = row["output_snr_db"] - row["input_snr_db"]
            gap = batch["x"][i, 1].numpy() < 0.5
            row["gap_rmse"] = float(np.sqrt(np.mean((prediction[i, gap] - target[i, gap]) ** 2))) if gap.any() else None
            rows.append(row)
    return rows


def summarize_metrics(rows):
    def average(group):
        result = {"count": len(group)}
        for key in group[0]:
            if key == "combo":
                continue
            values = [r[key] for r in group if r[key] is not None]
            result[key] = float(np.mean(values)) if values else None
        return result
    summary = {"overall": average(rows), "by_combination": {}}
    for combo in sorted({r["combo"] for r in rows}):
        label = "+".join(name for bit, name in enumerate(NOISE_NAMES) if combo & (1 << bit)) or "clean"
        summary["by_combination"][label] = average([r for r in rows if r["combo"] == combo])
    return summary


if RUN_EVALUATION:
    trained_model, checkpoint = load_trained_model(OUTPUT_DIR / "best.pt")
    eval_cfg, eval_noise = checkpoint["cfg"], checkpoint["noise_cfg"]
    with np.load(OUTPUT_DIR / "clean_banks.npz") as saved_banks:
        test_bank = saved_banks["test"].copy()
    test_set = MixedNoiseDataset(test_bank, "test", eval_cfg, eval_noise)
    metrics = evaluate_model(trained_model, make_loader(test_set, eval_cfg))
    summary = summarize_metrics(metrics)
    (OUTPUT_DIR / "test_metrics.json").write_text(json.dumps({"summary": summary, "records": metrics}, indent=2))
    print(json.dumps(summary, indent=2))

    example = test_set[0]
    noisy = example["x"][0].numpy() * float(example["scale"]) + float(example["offset"])
    valid = example["x"][1].numpy().astype(bool)
    target = example["y"][0].numpy() * float(example["scale"]) + float(example["offset"])
    restored = recover_signal(trained_model, noisy, valid)
    time = np.arange(len(target)) / eval_cfg["fs"]
    plt.figure(figsize=(12, 4))
    plt.plot(time, noisy, color="0.7", label="Input (interpolated at gaps)")
    plt.plot(time, target, "k--", linewidth=1, label="Clean target")
    plt.plot(time, restored, linewidth=1, label="TCN reconstruction")
    plt.fill_between(time, 0, 1, where=~valid, transform=plt.gca().get_xaxis_transform(),
                     color="orange", alpha=0.2, label="Missing samples")
    plt.xlabel("Time (s)"); plt.ylabel("Amplitude"); plt.legend(); plt.tight_layout(); plt.show()

# Inference on a real window sampled at checkpoint["cfg"]["fs"]:
# trained_model, checkpoint = load_trained_model(OUTPUT_DIR / "best.pt")
# raw = np.load("your_noisy_window.npy")  # 1-D; NaN marks known missing samples
# recovered = recover_signal(trained_model, raw)
# np.save("recovered_window.npy", recovered)